# 05 — Prompt Patterns and Technique Selection

Choose the smallest adequate system technique from measured failures—not from a catalog of fashionable patterns.

## Scenario, experimental question, and success criteria

An AI platform review board must assess 24 proposals. Candidate decisions include direct instruction, contrastive examples, schema constraints, retrieval, tools, bounded workflows, deterministic code, and no-model escalation.

**Experimental question.** Is topical pattern matching enough, or must deterministic alternatives, source authority, action authority, cost, and a frozen evaluation gate be checked first?

**Success criteria.** Compare three selectors on identical cases; report selection accuracy, unsafe-selection rate, avoidable-complexity rate, relative cost, estimated input tokens, and selector latency; demonstrate a relevant-looking technique that must be rejected.

## Learning objectives and safety boundaries

You will map observed failures to candidate techniques, test a simpler alternative first, define a disconfirming metric, and keep evidence and authorization boundaries outside model judgment.

The lab's relative cost units support comparison only. Metadata represents controls established by trusted application analysis; a model must never infer identity, permission, or tenant access from prose.

## Environment and reproducibility

The complete benchmark runs offline. The optional final call requires each learner's own `OPENAI_API_KEY`, `PROMPT_COURSE_PROVIDER=openai`, and explicit `RUN_LIVE=1`. Never paste a key into a notebook or commit it. One live proposal validates integration, not technique quality.

In [ ]:
from dataclasses import asdict
from importlib.util import module_from_spec, spec_from_file_location
from pathlib import Path
import os
import sys

import matplotlib.pyplot as plt
import pandas as pd

sys.path.insert(0, str(Path('src').resolve()))
module_path = Path('curriculum/beginner/05-prompt-patterns-and-technique-selection/lab.py')
spec = spec_from_file_location('course05_technique_lab', module_path)
lab = module_from_spec(spec)
sys.modules[spec.name] = lab
spec.loader.exec_module(lab)
print({'cases': len(lab.load_cases()), 'techniques': len(lab.TECHNIQUES)})

## Decision workflow

```text
observed failure → deterministic alternative? → authority/source checks
                → smallest candidate → frozen evaluation + budget
                → accept / reject / no model
```

A technique is a hypothesis about a failure. Its name does not prove suitability, safety, or value.

## Freeze the architecture cases

Expected choices are set before selectors run. The suite contains ordinary design decisions plus cost, safety, authority, and injection boundaries.

In [ ]:
cases = lab.load_cases()
case_frame = pd.DataFrame([{
    'id': case.id, 'slice': case.slice, 'expected': case.expected,
    'failure_type': case.metadata['failure_type']
} for case in cases])
display(pd.crosstab(case_frame['slice'], case_frame['expected']))
assert len(cases) == 24
assert {'safety', 'deterministic', 'cost', 'boundary'} <= set(case_frame['slice'])

## Inspect the technique contract

Each option names the failure it can address, maturity, relative cost, and an explicit avoid condition. `no_model` is a valid system decision when source or action authority is missing.

In [ ]:
pd.DataFrame([asdict(item) for item in lab.TECHNIQUES]).set_index('name')

## Baseline — maximalist architecture

The baseline chooses a planner/verifier workflow for every proposal. Hypothesis: it will solve the technique-name problem badly, introduce avoidable complexity, and remain unsafe when the correct answer is not to use a model.

In [ ]:
maximalist_rows = lab.run_strategy('maximalist')
maximalist_metrics = lab.metrics(maximalist_rows)
pd.Series(maximalist_metrics).round(4)

## Step 1 — pattern matching is a useful but incomplete control

The second selector maps an observed failure category to a standard technique. It improves accuracy, but deliberately ignores source authorization, effect authorization, and no-model boundaries.

In [ ]:
pattern_rows = lab.run_strategy('pattern_match')
pattern_metrics = lab.metrics(pattern_rows)
pd.DataFrame({'maximalist': maximalist_metrics, 'pattern_match': pattern_metrics}).T.round(4)

## Step 2 — apply system guardrails before technique selection

The guardrailed selector first chooses ordinary code for explicit rules, rejects missing authority or unauthorized sources, then applies the pattern map. These are application decisions, not prompt instructions.

In [ ]:
guardrailed_rows = lab.run_strategy('guardrailed')
guardrailed_metrics = lab.metrics(guardrailed_rows)
pd.Series(guardrailed_metrics).round(4)
assert guardrailed_metrics['unsafe_selection_rate'] == 0
assert guardrailed_metrics['avoidable_complexity_rate'] == 0

## Run the controlled selector comparison

Accuracy, safety, complexity, and relative cost answer different questions. A release gate should treat safety as a hard constraint rather than averaging it away.

In [ ]:
comparison = pd.DataFrame(lab.compare_strategies()).set_index('strategy')
comparison.round(4)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4.5), constrained_layout=True)
comparison[['selection_accuracy', 'unsafe_selection_rate', 'avoidable_complexity_rate']].plot.bar(ax=axes[0])
axes[0].set_ylim(0, 1.05); axes[0].set_ylabel('Rate'); axes[0].grid(axis='y', alpha=.25)
comparison['mean_cost_units'].plot.bar(ax=axes[1], color='#6C5CE7')
axes[1].set_ylabel('Relative cost units'); axes[1].set_title('Complexity cost'); axes[1].grid(axis='y', alpha=.25)
plt.show()

## Failure injection — relevant technique, prohibited system

A refund request appears to need live tool calling. The proposal explicitly lacks an authorization check. Topical pattern matching selects a tool; the correct system decision is `no_model` until an authorized application path exists.

In [ ]:
authority_case = next(case for case in cases if case.id == 'TEC-015')
injected = {strategy: lab.select(strategy, authority_case) for strategy in ('maximalist', 'pattern_match', 'guardrailed')}
assert injected['pattern_match'] == 'tool_calling'
assert injected['guardrailed'] == 'no_model'
injected

## Diagnose the failure

Pattern relevance answers “what might help?” It does not answer “may this system access the source or perform the effect?” Keep those as deterministic preconditions. The repair is not a stronger persona, another example, or an agent loop; it is an authorized application workflow or a rejected action.

In [ ]:
pd.DataFrame(lab.failure_matrix('pattern_match'))[[
    'case_id', 'slice', 'expected', 'selected', 'unsafe', 'avoidable_complexity'
]]

## Technology comparison

| Category | Stable default | Measure before adding |
| --- | --- | --- |
| foundational | direct contract, schema, validation, deterministic code | correctness and safe failure |
| practical | contrastive examples, approved retrieval, narrow tools | boundary gain, support, tool success, cost |
| model-dependent | planners, reflection loops, persona rituals | held-out end-to-end gain and stage failures |
| emerging | learned context policies, automated prompt/program search | leakage-resistant optimization and rollback |

No category overrides identity, permission, source governance, or effect authorization.

## Optional live provider implementation

The live adapter returns the same typed `SelectionResponse`. Export your own `OPENAI_API_KEY`, set `PROMPT_COURSE_PROVIDER=openai`, then set `RUN_LIVE=1` for one case. Compare the proposal with the deterministic guardrailed decision; do not let the model establish its own authority.

In [ ]:
if os.getenv('RUN_LIVE') == '1':
    if os.getenv('PROMPT_COURSE_PROVIDER') != 'openai' or not os.getenv('OPENAI_API_KEY'):
        raise RuntimeError('Set your own OPENAI_API_KEY and PROMPT_COURSE_PROVIDER=openai first.')
    provider_result = lab.run_provider_case(cases[0])
    display(provider_result.value.model_dump())
else:
    print('Skipped: the offline benchmark is complete; set RUN_LIVE=1 for one explicit integration call.')

## Production upgrade

| Notebook | Production |
| --- | --- |
| local case metadata | trusted policy/config services with named owners |
| relative cost units | measured model, retrieval, tool, and operations cost |
| one benchmark set | development, held-out, safety, and regression suites |
| local decision | review artifact with hypothesis, metric, reject rule, owner |
| printed failures | privacy-aware traces, dashboards, alerts, and rollback |

Retrieval requires source and tenant filters. Tools require authorization, validation, idempotency, and audit. Workflows require stage contracts, budgets, timeouts, stop conditions, and end-to-end gates.

## When not to use a prompt technique

Use deterministic code for explicit rules, conventional queries for structured lookups, and ordinary workflows when language judgment adds no value. Use `no_model` when identity, permission, evidence, or policy is absent. Complexity is justified only by a measured failure and a passing release gate.

## Review questions and exercises

1. Why is a topically relevant technique not necessarily a safe one?
2. Add a case where an authorized retrieval source is stale; define the expected decision.
3. Replace relative costs with measurements for an approved stack.
4. Define a gate that permits a bounded workflow only if it improves held-out quality within a latency budget.
5. Explain why schema validity cannot repair missing evidence.

**Advanced challenge.** Build a Pareto frontier for held-out quality, safety, latency, and cost. Treat authority as a hard constraint, quantify uncertainty, and test whether the frontier changes under distribution shift.

## Summary

Technique selection is a falsifiable system decision. Start from an observed failure, check deterministic and no-model paths, choose the smallest adequate candidate, define its metric and rejection rule, and release only when frozen quality, safety, latency, and cost evidence justify the added surface.